In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc

# Configuration
PROCESSED_DIR = Path('../processed')
OUTPUT_DIR = Path('../processed')
YEAR = 2022

# Columns needed from the parquet files
columns_needed = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'trip_distance',
    'PULocationID',
    'qa_flags'
]

files = sorted(PROCESSED_DIR.glob(f'yellow_tripdata_{YEAR}-*.parquet'))

# --- ACCUMULATORS ---
# We use lists to store small chunks of data, then combine them at the end.
# This avoids Index alignment errors and is faster than iterative addition.
zone_counts_parts = []     # For Top Zones
heatmap_counts_parts = []  # For Heatmap
duration_dow_data = []     # For Duration P95
speed_hour_data = []       # For Speed Median

print("Starting memory-efficient iterative processing...")

for f in files:
    print(f"Processing {f.name}...", end=" ")
    
    # 1. Load Chunk
    df_chunk = pd.read_parquet(f, columns=columns_needed)
    
    # 2. Filter (Keep only clean rows)
    df_chunk = df_chunk[df_chunk['qa_flags'] == 0].copy()
    
    if df_chunk.empty:
        print("Skipping (No clean data)")
        continue

    # 3. Calculate Derived Features
    # Duration
    df_chunk['trip_duration_minutes'] = (df_chunk['tpep_dropoff_datetime'] - df_chunk['tpep_pickup_datetime']).dt.total_seconds() / 60
    
    # Dimensions
    df_chunk['day_of_week'] = df_chunk['tpep_pickup_datetime'].dt.day_name()
    df_chunk['hour'] = df_chunk['tpep_pickup_datetime'].dt.hour
    
    # Speed
    df_chunk['speed_mph'] = np.where(
        df_chunk['trip_duration_minutes'] > 0,
        df_chunk['trip_distance'] / (df_chunk['trip_duration_minutes'] / 60),
        np.nan
    )
    
    # --- AGGREGATION 1: Top Zones (Count) ---
    # Store the value_counts series in the list
    zone_counts_parts.append(df_chunk['PULocationID'].value_counts())
    
    # --- AGGREGATION 2: Heatmap (Count) ---
    # Store the groupby size series in the list
    heatmap_counts_parts.append(df_chunk.groupby(['day_of_week', 'hour']).size())
    
    # --- ACCUMULATION 1: Duration Data (For P95 later) ---
    lean_duration = df_chunk[['day_of_week', 'trip_duration_minutes']].copy()
    duration_dow_data.append(lean_duration)
    
    # --- ACCUMULATION 2: Speed Data (For Median later) ---
    lean_speed = df_chunk.loc[df_chunk['speed_mph'] <= 100, ['hour', 'speed_mph']].copy()
    speed_hour_data.append(lean_speed)
    
    print(f"Done. (Rows: {len(df_chunk)})")
    
    # 4. Clean up Memory
    del df_chunk
    del lean_duration
    del lean_speed
    gc.collect()

print("\nAll files processed. Finalizing KPIs...")

# =========================================================
# KPI 1: P95 Duration by Day of Week
# =========================================================
print("Calculating P95 Duration...")
if duration_dow_data:
    df_duration_all = pd.concat(duration_dow_data, ignore_index=True)
    kpi_dow = df_duration_all.groupby('day_of_week')['trip_duration_minutes'].quantile(0.95).reset_index()
    kpi_dow.columns = ['Day_of_Week', 'p95_trip_duration']

    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    kpi_dow['Day_of_Week'] = pd.Categorical(kpi_dow['Day_of_Week'], categories=weekday_order, ordered=True)
    kpi_dow = kpi_dow.sort_values('Day_of_Week')

    kpi_dow.to_csv(OUTPUT_DIR / 'kpi_granular_dow_duration.csv', index=False)
    del df_duration_all
    del duration_dow_data
    gc.collect()

# =========================================================
# KPI 2: Median Speed by Hour
# =========================================================
print("Calculating Median Speed...")
if speed_hour_data:
    df_speed_all = pd.concat(speed_hour_data, ignore_index=True)
    kpi_speed = df_speed_all.groupby('hour')['speed_mph'].median().reset_index()
    kpi_speed.columns = ['hour_of_day', 'p50_speed_mph']
    kpi_speed.to_csv(OUTPUT_DIR / 'kpi_granular_speed_hour.csv', index=False)
    del df_speed_all
    del speed_hour_data
    gc.collect()

# =========================================================
# KPI 3: Top 10 Zones
# =========================================================
print("Finalizing Top Zones...")
if zone_counts_parts:
    # Concatenate all partial counts, group by index (PULocationID), and sum
    total_zone_counts = pd.concat(zone_counts_parts).groupby(level=0).sum()
    
    # Get top 10
    top_zones = total_zone_counts.nlargest(10).reset_index()
    top_zones.columns = ['PULocationID', 'trip_count']
    
    # Load lookup and merge
    df_lookup = pd.read_csv('../raw/taxi_zone_lookup.csv')
    df_lookup = df_lookup.rename(columns={'LocationID': 'PULocationID', 'Zone': 'pickup_zone'})
    
    top_zones = top_zones.merge(df_lookup[['PULocationID', 'pickup_zone']], on='PULocationID', how='left')
    top_zones.to_csv(OUTPUT_DIR / 'kpi_granular_top_zones.csv', index=False)
    del zone_counts_parts

# =========================================================
# KPI 4: Heatmap Data
# =========================================================
print("Finalizing Heatmap...")
if heatmap_counts_parts:
    # Concatenate all partial MultiIndex series, group by levels (Day, Hour), and sum
    total_heatmap_counts = pd.concat(heatmap_counts_parts).groupby(level=[0, 1]).sum()
    
    heatmap_df = total_heatmap_counts.reset_index(name='trip_count')
    heatmap_df.to_csv(OUTPUT_DIR / 'kpi_granular_heatmap.csv', index=False)
    del heatmap_counts_parts

print("Done. All granular KPIs saved successfully.")

Starting memory-efficient iterative processing...
Processing yellow_tripdata_2022-01.parquet... Done. (Rows: 1661330)
Processing yellow_tripdata_2022-02.parquet... Done. (Rows: 1998438)
Processing yellow_tripdata_2022-03.parquet... Done. (Rows: 2454935)
Processing yellow_tripdata_2022-04.parquet... Done. (Rows: 2426704)
Processing yellow_tripdata_2022-05.parquet... Done. (Rows: 2382965)
Processing yellow_tripdata_2022-06.parquet... Done. (Rows: 2369755)
Processing yellow_tripdata_2022-07.parquet... Done. (Rows: 2145032)
Processing yellow_tripdata_2022-08.parquet... Done. (Rows: 2162803)
Processing yellow_tripdata_2022-09.parquet... Done. (Rows: 2143466)
Processing yellow_tripdata_2022-10.parquet... Done. (Rows: 2471895)
Processing yellow_tripdata_2022-11.parquet... Done. (Rows: 2374193)
Processing yellow_tripdata_2022-12.parquet... Done. (Rows: 2328151)

All files processed. Finalizing KPIs...
Calculating P95 Duration...
Calculating Median Speed...
Finalizing Top Zones...
Finalizing He